In [1]:
import os, re, json
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import fitz  # PyMuPDF
from pathlib import Path


In [2]:
# -------------------------------------------------------------------
# Patterns that MUST be preserved for downstream enrichment
# -------------------------------------------------------------------
ANCHOR_PATTERNS = [
    r'^\s*(?:CORAM)\b[:\s-].*',
    r'^\s*(?:BENCH)\b[:\s-].*',
    r'^\s*(?:JUDGMENT|JUDGEMENT)\b.*',
    r'^\s*(?:ORDER)\b.*',
    r'^\s*(?:FACTS|BACKGROUND|BRIEF FACTS)\b.*',
    r'^\s*(?:ISSUES|ARGUMENTS|ANALYSIS|REASONING|CONCLUSION)\b.*',
]

# -------------------------------------------------------------------
# Citation patterns we want to PRESERVE
# -------------------------------------------------------------------
CITATION_INLINE_PATTERNS = [
    r'\[\d{4}\]\s*\d+\s*S\.C\.R\.\s*\d+',
    r'\(\d{4}\)\s*\d+\s*SCC\s*\d+',
    r'\bAIR\s+\d{4}\b',
    r'\bSCC\b',
    r'\bSCR\b',
]

def contains_citation(line):
    return any(re.search(p, line) for p in CITATION_INLINE_PATTERNS)


# -------------------------------------------------------------------
# True page-header patterns (only delete these, not real citations!)
# -------------------------------------------------------------------
PAGE_HEADER_PATTERNS = [
    r'^\s*SUPREME COURT REPORTS\b.*$',
    r'^\s*SUPREME COURT OF INDIA\b.*$',
    r'^\s*ITEM NO\..*$',
    r'^\s*COURT NO\..*$',
    r'^\s*SECTION\s+[A-Z0-9-]+$',
    r'^\s*REPORTABLE\s*$',
    r'^\s*NON-REPORTABLE\s*$',
]

PAGE_NUMBER_PATTERN = re.compile(r'^\s*(?:Page\s+)?\d+\s*$', flags=re.IGNORECASE)


# -------------------------------------------------------------------
# CLEANING FUNCTION — FINAL VERSION (STRUCTURE PRESERVED)
# -------------------------------------------------------------------
def clean_case_text_keep_structure(text):

    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = text.split('\n')

    cleaned = []

    for ln in lines:
        raw_line = ln.strip()

        if raw_line == "":
            cleaned.append("")
            continue

        # 1. KEEP all anchor lines
        if any(re.match(p, ln, flags=re.IGNORECASE) for p in ANCHOR_PATTERNS):
            cleaned.append(raw_line)
            continue

        # 2. Keep real citation lines
        if contains_citation(ln):
            cleaned.append(raw_line)
            continue

        # 3. Remove pure page numbers ONLY
        if PAGE_NUMBER_PATTERN.match(ln):
            continue

        # 4. Remove TRUE page headers (but NOT citations!)
        if any(re.match(p, ln, flags=re.IGNORECASE) for p in PAGE_HEADER_PATTERNS):
            continue

        # 5. Remove long dashed separators
        if re.match(r'^\s*[-_]{5,}\s*$', ln):
            continue

        # 6. Normalize spacing inside lines (NOT newlines)
        ln2 = re.sub(r'[ \t]{2,}', ' ', ln).rstrip()
        cleaned.append(ln2)

    # ---------------------------------------------
    # Preserve paragraph structure
    # ---------------------------------------------
    joined = "\n".join(cleaned)
    joined = re.sub(r'\n{3,}', '\n\n', joined)  # collapse >2 blank lines

    # Remove insane letter ladders like A B C D ...
    joined = re.sub(r'(^|\n)(?:[A-Z]\s+){5,}(\n|$)', '\n', joined)

    # Final trim
    return joined.strip()


In [3]:
def extract_light_metadata(text):
    meta = {}

    paragraphs = [p for p in re.split(r'\n\s*\n', text) if p.strip()]
    meta['paragraph_count'] = len(paragraphs)
    meta['word_count'] = sum(len(p.split()) for p in paragraphs)
    meta['first_paragraphs'] = paragraphs[:2]

    # Find judge/bench lines
    def find(pattern):
        m = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)
        return m.group(0).strip() if m else None

    meta['coram_line'] = find(r'^.*\bCORAM\b.*$')
    meta['bench_line'] = find(r'^.*\bBENCH\b.*$')
    meta['has_judgment_header'] = bool(find(r'^.*\bJUDGMENT\b.*$'))

    return meta


In [4]:
def extract_text_fitz(pdf_path, txt_path, meta_path=None):

    try:
        pages = []
        with fitz.open(pdf_path) as doc:
            for pg in doc:
                txt = pg.get_text("text")
                # fallback
                if not txt.strip():
                    txt = "\n".join(b[4] for b in pg.get_text("blocks") if b[4].strip())

                pages.append(txt.strip())

        raw = "\n\n".join(pages)

        cleaned = clean_case_text_keep_structure(raw)

        # Fallback — ensure text is not empty after cleaning
        if len(cleaned) < 500:
            cleaned = raw

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(cleaned)

        if meta_path:
            meta = extract_light_metadata(cleaned)
            meta['source_pdf'] = os.path.basename(pdf_path)
            meta['txt_file'] = os.path.basename(txt_path)

            with open(meta_path, "w", encoding="utf-8") as f:
                json.dump(meta, f, indent=2, ensure_ascii=False)

    except Exception as e:
        print(f"⚠️ Error: {pdf_path} → {e}")


In [5]:
START = int(input("Enter Start Year: "))
END   = int(input("Enter End Year: "))

BASE_DIR = f"D:/LPA_MTech_Project/My_Datasets/SC_{START}-{END}"
OUT_DIR  = f"D:/LPA_MTech_Project/Extracted_Texts/Texts_{START}-{END}"

os.makedirs(OUT_DIR, exist_ok=True)


In [6]:
def collect_pdfs(base_dir, start, end):
    pdfs = []
    for yr in range(start, end + 1):
        year_path = os.path.join(base_dir, str(yr), "english")
        if os.path.exists(year_path):
            found = [
                os.path.join(year_path, f)
                for f in os.listdir(year_path)
                if f.lower().endswith(".pdf")
            ]
            pdfs.extend(found)
            print(f"[{yr}] Found {len(found)} PDFs")
    print("TOTAL:", len(pdfs))
    return pdfs

pdf_files = collect_pdfs(BASE_DIR, START, END)


[2010] Found 907 PDFs
[2011] Found 848 PDFs
[2012] Found 623 PDFs
[2013] Found 853 PDFs
[2014] Found 795 PDFs
[2015] Found 760 PDFs
[2016] Found 589 PDFs
[2017] Found 732 PDFs
[2018] Found 795 PDFs
[2019] Found 1050 PDFs
TOTAL: 7952


In [7]:
with ThreadPoolExecutor(max_workers=6) as ex:
    for p in tqdm(pdf_files, desc="Extracting PDFs"):
        base = Path(p).stem
        txt_path = os.path.join(OUT_DIR, base + ".txt")
        meta_path = os.path.join(OUT_DIR, base + ".meta.json")
        ex.submit(extract_text_fitz, p, txt_path, meta_path)

print("✔ DONE — Text + metadata saved to:", OUT_DIR)


Extracting PDFs: 100%|██████████| 7952/7952 [00:25<00:00, 310.49it/s] 


✔ DONE — Text + metadata saved to: D:/LPA_MTech_Project/Extracted_Texts/Texts_2010-2019


In [8]:
sample = Path(pdf_files[0]).stem

print("TXT PREVIEW:\n")
with open(os.path.join(OUT_DIR, sample + ".txt"), "r", encoding="utf-8") as f:
    print(f.read()[:2000])


TXT PREVIEW:

A
B
c
D
E
[2010] 10 S.C.R. 1002
SWAMI NATH
v.
NIRMAL SINGH
(SLP (Civil) No. 8317 of 2006)
SEPTEMBER 7, 2010
[ALTAMAS KABIR, A.K. PATNAIK AND ANIL
R. DAVE, JJ.]
Rent Control and Eviction:
East Punjab Urban Rent Restriction Act, 1949:
s.13-8 rlw. s.18-A - Eviction Petition - By non-resident
Indian - Allowed by courts below - On appeal, held: Landlord
entitled to eviction.
s. 13-8 (As amended in the year 2001) - Interpretation
of - Held: Interpretation of the provision that right of
immediate possession can be exercised only once, would
frustrate the object of the amendment to the provision.
The respondent-landlords who were non-resident
Indians, filed petitions u/s. 13-B of East Punjab Urban
Rent Restriction Act, 1949. They sought eviction of
tenants from their respective tenanted premises. The
F Rent Controller allowed all the three eviction petitions.
The petitioners-tenants moved the High Court in revision
petition. In all the revision petitions, the common plea was
that

In [9]:
with open(os.path.join(OUT_DIR, sample + ".meta.json"), "r", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2))


{
  "paragraph_count": 7,
  "word_count": 2169,
  "first_paragraphs": [
    "A\nB\nc\nD\nE\n[2010] 10 S.C.R. 1002\nSWAMI NATH\nv.\nNIRMAL SINGH\n(SLP (Civil) No. 8317 of 2006)\nSEPTEMBER 7, 2010\n[ALTAMAS KABIR, A.K. PATNAIK AND ANIL\nR. DAVE, JJ.]\nRent Control and Eviction:\nEast Punjab Urban Rent Restriction Act, 1949:\ns.13-8 rlw. s.18-A - Eviction Petition - By non-resident\nIndian - Allowed by courts below - On appeal, held: Landlord\nentitled to eviction.\ns. 13-8 (As amended in the year 2001) - Interpretation\nof - Held: Interpretation of the provision that right of\nimmediate possession can be exercised only once, would\nfrustrate the object of the amendment to the provision.\nThe respondent-landlords who were non-resident\nIndians, filed petitions u/s. 13-B of East Punjab Urban\nRent Restriction Act, 1949. They sought eviction of\ntenants from their respective tenanted premises. The\nF Rent Controller allowed all the three eviction petitions.\nThe petitioners-tenants moved th